In [1]:
%matplotlib qt

In [17]:
import sys
from pathlib import Path

# ============================================
# PROJECT PATH SETUP
# ============================================

PROJECT_ROOT = Path.cwd().parents[1]   # go from notebooks → Python
SRC_PATH = PROJECT_ROOT / "Python" / "src"

sys.path.append(str(SRC_PATH))

print("Added to path:", SRC_PATH)

repo_root = Path.cwd().parent
sys.path.insert(0, str(repo_root))

print("Added to path:", repo_root)


# ============================================
# AIR WHEEL — MP4 DIRECTORY SCANNER
# NeuroMomentum Lab
# ============================================

from pathlib import Path
import json

# --- Windows local data path ---
ROOT = Path(r"E:\Data\UNLV\AIR_Wheel_Methods")

# --- animals and dates of interest ---
# TARGETS = {
#     "NML_M_08": "2026_03_04",
# }
animals = ["NML_GC_01_R","NML_04_R","NML_05_R","NML_06_R"]
dates = {
    "NML_GC_01_R": "2025_12_16",
    "NML_04_R": "2026_01_24",
    "NML_05_R": "2026_01_14",
    "NML_06_R": "2026_01_16",
}

Added to path: g:\My Drive\Research\GitHub\AIR_Wheel_Methods\Python\src
Added to path: g:\My Drive\Research\GitHub\AIR_Wheel_Methods\Python


In [20]:
print(animal)

NML_GC_01_R


In [33]:
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path(r"E:\Data\UNLV\AIR_Wheel_Methods\PData")

an = 3
animal = animals[an]
date = dates[animal]

session_dir = ROOT / animal / date

videos = list(session_dir.glob("*.mp4"))

print("Videos found:")
for v in videos:
    print(v.name)

Videos found:
face_1440x1080_60_20260116_154145.mp4
pupi_320x240_60_20260116_154145.mp4
video_20260116_154140.mp4


In [4]:
import cv2
import json
from pathlib import Path

def select_and_save_roi(video_path):

    cap = cv2.VideoCapture(str(video_path))
    ret, frame = cap.read()
    cap.release()

    if not ret:
        raise RuntimeError("Could not read video frame")

    roi = cv2.selectROI("Select LED ROI", frame, fromCenter=False)
    cv2.destroyAllWindows()

    x, y, w, h = roi

    roi_dict = {
        "x1": int(x),
        "y1": int(y),
        "x2": int(x + w),
        "y2": int(y + h)
    }

    json_path = video_path.with_name(video_path.stem + "_LED_ROI.json")

    with open(json_path, "w") as f:
        json.dump(roi_dict, f, indent=4)

    print("ROI saved:", json_path)

    return roi_dict


def load_roi(video_path):

    json_path = video_path.with_name(video_path.stem + "_LED_ROI.json")

    with open(json_path, "r") as f:
        roi = json.load(f)

    return roi

In [34]:
for video in videos:
    roi = select_and_save_roi(video)
    print("ROI:", roi)

ROI saved: E:\Data\UNLV\AIR_Wheel_Methods\PData\NML_06_R\2026_01_16\face_1440x1080_60_20260116_154145_LED_ROI.json
ROI: {'x1': 80, 'y1': 326, 'x2': 114, 'y2': 357}
ROI saved: E:\Data\UNLV\AIR_Wheel_Methods\PData\NML_06_R\2026_01_16\pupi_320x240_60_20260116_154145_LED_ROI.json
ROI: {'x1': 119, 'y1': 21, 'x2': 131, 'y2': 32}
ROI saved: E:\Data\UNLV\AIR_Wheel_Methods\PData\NML_06_R\2026_01_16\video_20260116_154140_LED_ROI.json
ROI: {'x1': 871, 'y1': 40, 'x2': 892, 'y2': 68}


In [35]:
import numpy as np
import pandas as pd

def extract_led_and_save(video_path, roi):

    x1 = roi["x1"]
    y1 = roi["y1"]
    x2 = roi["x2"]
    y2 = roi["y2"]

    cap = cv2.VideoCapture(str(video_path))

    fps = cap.get(cv2.CAP_PROP_FPS)

    if fps == 0:
        raise RuntimeError("Could not read FPS from video")

    print("Video:", video_path.name, "FPS:", fps)

    frame_idx = 0

    frames = []
    times = []
    times_ms = []

    mean_r = []
    mean_g = []
    mean_b = []
    red_strength = []

    while True:

        ret, frame = cap.read()
        if not ret:
            break

        roi_frame = frame[y1:y2, x1:x2]

        b,g,r = cv2.split(roi_frame)

        r_mean = r.mean()
        g_mean = g.mean()
        b_mean = b.mean()

        strength = r_mean - (g_mean + b_mean)/2

        frames.append(frame_idx)
        times.append(frame_idx / fps)
        time_ms = cap.get(cv2.CAP_PROP_POS_MSEC)
        times_ms.append(time_ms)

        mean_r.append(r_mean)
        mean_g.append(g_mean)
        mean_b.append(b_mean)
        red_strength.append(strength)

        frame_idx += 1

    cap.release()

    red_strength = np.array(red_strength)

    # ---------- adaptive threshold ----------
    lo = np.percentile(red_strength,20)
    hi = np.percentile(red_strength,80)

    thr_on = lo + 0.6*(hi-lo)
    thr_off = lo + 0.4*(hi-lo)

    led_state = []
    state = False
    state = red_strength[0] >= thr_on

    for val in red_strength:

        if not state and val >= thr_on:
            state = True
        elif state and val <= thr_off:
            state = False

        led_state.append(state)

    # ---------- dataframe ----------
    df = pd.DataFrame({
        "frame": frames,
        "time_sec": times,
        "time_ms": times_ms,
        "fps": fps,
        "mean_red": mean_r,
        "mean_green": mean_g,
        "mean_blue": mean_b,
        "red_strength": red_strength,
        "threshold_on": thr_on,
        "threshold_off": thr_off,
        "LED_on": led_state
    })

    csv_path = video_path.with_name(video_path.stem + "_LED_signal.csv")

    df.to_csv(csv_path, index=False)

    print("LED signal saved:", csv_path)

    return df

In [41]:
import numpy as np
import pandas as pd
import cv2

def extract_led_and_save(
    video_path,
    roi,
    smooth_window=31,
    min_on_sec=1.0,
    min_off_sec=2.0,
    use_grayscale=True,
):
    """
    Extract LED signal from ROI, smooth it, threshold it, and remove short glitches.

    Parameters
    ----------
    video_path : pathlib.Path or str
        Input video path
    roi : dict
        ROI with keys x1, y1, x2, y2
    smooth_window : int
        Median filter window size (odd integer recommended)
    min_on_sec : float
        Minimum ON duration to keep (seconds)
    min_off_sec : float
        Minimum OFF duration to keep (seconds)
    use_grayscale : bool
        If True, use grayscale mean intensity as LED signal.
        If False, use red-strength = R - (G+B)/2.

    Returns
    -------
    df : pandas.DataFrame
        Table containing raw signal, smoothed signal, thresholds, and cleaned LED state
    """

    x1 = roi["x1"]
    y1 = roi["y1"]
    x2 = roi["x2"]
    y2 = roi["y2"]

    cap = cv2.VideoCapture(str(video_path))
    fps = cap.get(cv2.CAP_PROP_FPS)

    if fps == 0:
        raise RuntimeError("Could not read FPS from video")

    print("Video:", video_path.name, "FPS:", fps)

    frame_idx = 0
    frames = []
    times = []
    times_ms = []

    mean_r = []
    mean_g = []
    mean_b = []
    signal_raw = []

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        roi_frame = frame[y1:y2, x1:x2]

        b, g, r = cv2.split(roi_frame)

        r_mean = r.mean()
        g_mean = g.mean()
        b_mean = b.mean()

        if use_grayscale:
            gray = cv2.cvtColor(roi_frame, cv2.COLOR_BGR2GRAY)
            strength = gray.mean()
        else:
            strength = r_mean - (g_mean + b_mean) / 2.0

        frames.append(frame_idx)
        times.append(frame_idx / fps)
        times_ms.append(cap.get(cv2.CAP_PROP_POS_MSEC))

        mean_r.append(r_mean)
        mean_g.append(g_mean)
        mean_b.append(b_mean)
        signal_raw.append(strength)

        frame_idx += 1

    cap.release()

    signal_raw = np.asarray(signal_raw, dtype=float)

    # -----------------------------------
    # Median smoothing
    # -----------------------------------
    if smooth_window < 1:
        smooth_window = 1
    if smooth_window % 2 == 0:
        smooth_window += 1  # make odd

    signal_s = pd.Series(signal_raw).rolling(
        window=smooth_window, center=True, min_periods=1
    ).median().to_numpy()

    # -----------------------------------
    # Thresholding
    # Simpler threshold works better for your signal
    # -----------------------------------
    thr = signal_s.mean()

    led_state_raw = signal_s > thr

    # -----------------------------------
    # Remove short ON glitches
    # -----------------------------------
    min_on_frames = max(1, int(round(min_on_sec * fps)))
    min_off_frames = max(1, int(round(min_off_sec * fps)))

    led_state = _remove_short_runs(led_state_raw, keep_value=True, min_len=min_on_frames)
    led_state = _remove_short_runs(led_state, keep_value=False, min_len=min_off_frames)

    # -----------------------------------
    # Save
    # -----------------------------------
    df = pd.DataFrame({
        "frame": frames,
        "time_sec": times,
        "time_ms": times_ms,
        "fps": fps,
        "mean_red": mean_r,
        "mean_green": mean_g,
        "mean_blue": mean_b,
        "signal_raw": signal_raw,
        "signal_smooth": signal_s,
        "threshold": thr,
        "LED_on_raw": led_state_raw,
        "LED_on": led_state,
    })

    csv_path = video_path.with_name(video_path.stem + "_LED_signal.csv")
    df.to_csv(csv_path, index=False)

    print("LED signal saved:", csv_path)
    print(f"Total detected ON events (cleaned): {_count_onsets(led_state)}")

    return df


def _remove_short_runs(arr, keep_value=True, min_len=1):
    """
    Remove runs shorter than min_len.
    If keep_value=True, removes short True runs.
    If keep_value=False, removes short False runs (fills small gaps).
    """
    arr = np.asarray(arr, dtype=bool).copy()
    target = keep_value

    n = len(arr)
    i = 0
    while i < n:
        if arr[i] == target:
            j = i
            while j < n and arr[j] == target:
                j += 1
            run_len = j - i
            if run_len < min_len:
                arr[i:j] = ~target
            i = j
        else:
            i += 1
    return arr


def _count_onsets(arr):
    arr = np.asarray(arr, dtype=bool)
    return np.sum((~arr[:-1]) & (arr[1:])) + int(arr[0])

In [42]:

for animal in animals:
    # animal = animals[an]
    date = dates[animal]

    session_dir = ROOT / animal / date

    videos = list(session_dir.glob("*.mp4"))

    print("Videos found:")
    for v in videos:
        print(v.name)

    for video in videos:

        print("Processing:", video.name)

        roi = load_roi(video)

        df = extract_led_and_save(video, roi)

Videos found:
face_1440x1080_60_20251216_165824.mp4
pupi_320x240_60_20251216_165824.mp4
video_20251216_165824.mp4
Processing: face_1440x1080_60_20251216_165824.mp4
Video: face_1440x1080_60_20251216_165824.mp4 FPS: 60.243020336419825
LED signal saved: E:\Data\UNLV\AIR_Wheel_Methods\PData\NML_GC_01_R\2025_12_16\face_1440x1080_60_20251216_165824_LED_signal.csv
Total detected ON events (cleaned): 34
Processing: pupi_320x240_60_20251216_165824.mp4
Video: pupi_320x240_60_20251216_165824.mp4 FPS: 62.41163855351009
LED signal saved: E:\Data\UNLV\AIR_Wheel_Methods\PData\NML_GC_01_R\2025_12_16\pupi_320x240_60_20251216_165824_LED_signal.csv
Total detected ON events (cleaned): 34
Processing: video_20251216_165824.mp4
Video: video_20251216_165824.mp4 FPS: 60.5282054234884
LED signal saved: E:\Data\UNLV\AIR_Wheel_Methods\PData\NML_GC_01_R\2025_12_16\video_20251216_165824_LED_signal.csv
Total detected ON events (cleaned): 34
Videos found:
face_1440x1080_60_20260124_154254.mp4
pupi_320x240_60_20260124

C:\Users\inayas1\AppData\Local\Temp\ipykernel_86444\1773665444.py:169: DeprecationWarning: Bitwise inversion '~' on bool is deprecated and will be removed in Python 3.16. This returns the bitwise inversion of the underlying int object and is usually not what you expect from negating a bool. Use the 'not' operator for boolean negation or ~int(x) if you really want the bitwise inversion of the underlying int.
  arr[i:j] = ~target


LED signal saved: E:\Data\UNLV\AIR_Wheel_Methods\PData\NML_04_R\2026_01_24\video_20260124_154256_LED_signal.csv
Total detected ON events (cleaned): 54
Videos found:
face_1440x1080_60_20260114_164026.mp4
pupi_320x240_60_20260114_164026.mp4
video_20260114_164026.mp4
Processing: face_1440x1080_60_20260114_164026.mp4
Video: face_1440x1080_60_20260114_164026.mp4 FPS: 51.60145501414362
LED signal saved: E:\Data\UNLV\AIR_Wheel_Methods\PData\NML_05_R\2026_01_14\face_1440x1080_60_20260114_164026_LED_signal.csv
Total detected ON events (cleaned): 58
Processing: pupi_320x240_60_20260114_164026.mp4
Video: pupi_320x240_60_20260114_164026.mp4 FPS: 62.173203942691494
LED signal saved: E:\Data\UNLV\AIR_Wheel_Methods\PData\NML_05_R\2026_01_14\pupi_320x240_60_20260114_164026_LED_signal.csv
Total detected ON events (cleaned): 58
Processing: video_20260114_164026.mp4
Video: video_20260114_164026.mp4 FPS: 62.35504530006994
LED signal saved: E:\Data\UNLV\AIR_Wheel_Methods\PData\NML_05_R\2026_01_14\video_202

In [40]:
for animal in animals:
    # animal = animals[an]
    date = dates[animal]

    session_dir = ROOT / animal / date

    videos = list(session_dir.glob("*.mp4"))

    print("Videos found:")
    for v in videos:
        print(v.name)


Videos found:
face_1440x1080_60_20251216_165824.mp4
pupi_320x240_60_20251216_165824.mp4
video_20251216_165824.mp4
Videos found:
face_1440x1080_60_20260124_154254.mp4
pupi_320x240_60_20260124_154254.mp4
video_20260124_154256.mp4
Videos found:
face_1440x1080_60_20260114_164026.mp4
pupi_320x240_60_20260114_164026.mp4
video_20260114_164026.mp4
Videos found:
face_1440x1080_60_20260116_154145.mp4
pupi_320x240_60_20260116_154145.mp4
video_20260116_154140.mp4


In [36]:
print(videos[2])

E:\Data\UNLV\AIR_Wheel_Methods\PData\NML_GC_01_R\2025_12_16\video_20251216_165824.mp4


In [78]:
video = videos[1]
roi = load_roi(video)

df = extract_led_and_save(video, roi)

Video: pupi_320x240_60_20260116_154145.mp4 FPS: 60.90751441794699
LED signal saved: E:\Data\UNLV\AIR_Wheel_Methods\PData\NML_06_R\2026_01_16\pupi_320x240_60_20260116_154145_LED_signal.csv


In [76]:
import numpy as np
import pandas as pd
import cv2

def extract_led_intensity_and_save(video_path, roi):

    x1 = roi["x1"]
    y1 = roi["y1"]
    x2 = roi["x2"]
    y2 = roi["y2"]

    cap = cv2.VideoCapture(str(video_path))

    fps = cap.get(cv2.CAP_PROP_FPS)

    if fps == 0:
        raise RuntimeError("Could not read FPS from video")

    print("Video:", video_path.name, "FPS:", fps)

    frame_idx = 0

    frames = []
    times = []
    times_ms = []

    mean_r = []
    mean_g = []
    mean_b = []
    intensity_strength = []

    while True:

        ret, frame = cap.read()
        if not ret:
            break

        roi_frame = frame[y1:y2, x1:x2]

        # split channels (for saving same columns)
        b, g, r = cv2.split(roi_frame)

        r_mean = r.mean()
        g_mean = g.mean()
        b_mean = b.mean()

        # compute grayscale intensity
        gray = cv2.cvtColor(roi_frame, cv2.COLOR_BGR2GRAY)
        intensity = gray.mean()

        frames.append(frame_idx)
        times.append(frame_idx / fps)

        time_ms = cap.get(cv2.CAP_PROP_POS_MSEC)
        times_ms.append(time_ms)

        mean_r.append(r_mean)
        mean_g.append(g_mean)
        mean_b.append(b_mean)

        intensity_strength.append(intensity)

        frame_idx += 1

    cap.release()

    intensity_strength = np.array(intensity_strength)

    # ---------- adaptive threshold ----------
    lo = np.percentile(intensity_strength, 20)
    hi = np.percentile(intensity_strength, 80)

    thr_on = lo + 0.6 * (hi - lo)
    thr_off = lo + 0.4 * (hi - lo)

    led_state = []
    state = intensity_strength[0] >= thr_on

    for val in intensity_strength:

        if not state and val >= thr_on:
            state = True
        elif state and val <= thr_off:
            state = False

        led_state.append(state)

    # ---------- dataframe ----------
    df = pd.DataFrame({
        "frame": frames,
        "time_sec": times,
        "time_ms": times_ms,
        "fps": fps,
        "mean_red": mean_r,
        "mean_green": mean_g,
        "mean_blue": mean_b,
        "red_strength": intensity_strength,  # keep same column name
        "threshold_on": thr_on,
        "threshold_off": thr_off,
        "LED_on": led_state
    })

    csv_path = video_path.with_name(video_path.stem + "_LED_signal.csv")

    df.to_csv(csv_path, index=False)

    print("LED signal saved:", csv_path)

    return df

In [ ]:
video = videos[1]
roi = load_roi(video)

df = extract_led_intensity_and_save(video, roi)

Video: video_20251216_165824.mp4 FPS: 60.5282054234884
LED signal saved: E:\Data\UNLV\AIR_Wheel_Methods\PData\NML_GC_01_R\2025_12_16\video_20251216_165824_LED_signal.csv
